In [15]:

# ============================================================
# NOTEBOOK 10 — RECOGNITION WITH AUGMENTED DATA
# NDCLD15 | Exploratory proxy-label experiment only
# ============================================================

from pathlib import Path
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json, time, random, shutil, zipfile, hashlib
from collections import defaultdict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT = Path("/kaggle/working/nd_iris_project")
DATASET_DIR = PROJECT / "notebook09_dataset_augmentation/dataset"
OUT = PROJECT / "notebook10_recognition_augmented"
OUT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
EPOCHS = 3
LR = 1e-3
IMAGE_SIZE = 128
NUM_WORKERS = 2

print("Device:", DEVICE)
print("GPU count:", torch.cuda.device_count())

# ------------------------------------------------------------
# 1. Find real images and derive explicitly unverified proxy IDs
# ------------------------------------------------------------
real_dir = DATASET_DIR / "real"
trad_dir = DATASET_DIR / "traditional"
gan_dir = DATASET_DIR / "gan"
diff_dir = DATASET_DIR / "diffusion"

EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

def list_images(folder):
    if not folder.exists():
        return []
    return sorted(
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in EXTS
    )

real_files = list_images(real_dir)
trad_files = list_images(trad_dir)
gan_files = list_images(gan_dir)
diff_files = list_images(diff_dir)

print("Real:", len(real_files))
print("Traditional:", len(trad_files))
print("GAN:", len(gan_files))
print("Diffusion:", len(diff_files))

if len(real_files) < 20:
    raise RuntimeError("Insufficient real images for an exploratory split.")

# The consolidated filenames are hash-based, so use the original source
# filenames from the manifest to recover a filename-prefix proxy label.
manifest_path = PROJECT / "notebook09_dataset_augmentation/dataset_manifest.csv"
manifest = pd.read_csv(manifest_path)

copied_real = manifest[
    (manifest["source"] == "real") &
    (manifest["status"] == "copied")
].copy()

# Proxy label rule inherited from the earlier baseline:
# take the original filename prefix before the first 'd'.
# This is NOT a verified identity or contact-lens class label.
def proxy_label_from_original_path(path):
    name = Path(str(path)).name
    stem = Path(name).stem
    if "d" in stem:
        return stem.split("d", 1)[0]
    return None

copied_real["proxy_label"] = copied_real["original_path"].apply(
    proxy_label_from_original_path
)

label_map = dict(zip(
    copied_real["dataset_path"].astype(str),
    copied_real["proxy_label"]
))

records = []
for p in real_files:
    relative = str(p.relative_to(OUT.parent / "notebook09_dataset_augmentation"))
    label = label_map.get(relative)
    if label is not None and str(label).strip() != "":
        records.append({"path": str(p), "proxy_label": str(label)})

real_df = pd.DataFrame(records)

if real_df.empty:
    raise RuntimeError(
        "Could not recover proxy labels from the manifest. "
        "Do not substitute synthetic labels; inspect Notebook 09 manifest."
    )

# Keep only proxy labels with at least 2 images to permit a split.
counts = real_df["proxy_label"].value_counts()
usable_labels = counts[counts >= 2].index
real_df = real_df[real_df["proxy_label"].isin(usable_labels)].copy()

classes = sorted(real_df["proxy_label"].unique())
class_to_idx = {label: i for i, label in enumerate(classes)}
real_df["label"] = real_df["proxy_label"].map(class_to_idx)

print("Real images with proxy labels:", len(real_df))
print("Proxy classes:", len(classes))
print("WARNING: these labels are unverified filename-derived proxies.")

# ------------------------------------------------------------
# 2. Stratified per-class split; no augmentation variants in test
# ------------------------------------------------------------
train_rows, val_rows, test_rows = [], [], []

for label, group in real_df.groupby("label"):
    paths = group.to_dict("records")
    random.Random(SEED + int(label)).shuffle(paths)

    n = len(paths)
    if n < 2:
        continue

    n_test = max(1, int(round(0.15 * n)))
    n_val = max(1, int(round(0.15 * n))) if n >= 4 else 0

    # Preserve at least one training sample per class.
    while n_test + n_val >= n:
        if n_val > 0:
            n_val -= 1
        else:
            n_test -= 1

    test_rows.extend(paths[:n_test])
    val_rows.extend(paths[n_test:n_test+n_val])
    train_rows.extend(paths[n_test+n_val:])

train_df = pd.DataFrame(train_rows)
val_df = pd.DataFrame(val_rows)
test_df = pd.DataFrame(test_rows)

if train_df.empty or test_df.empty:
    raise RuntimeError("Split produced an empty train or test set.")

print("\nSplit sizes:")
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

# ------------------------------------------------------------
# 3. Dataset and transforms
# ------------------------------------------------------------
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.10, contrast=0.10),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

class OcularDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        with Image.open(row["path"]) as im:
            image = im.convert("RGB")
        image = self.transform(image)
        return image, int(row["label"])

train_ds = OcularDataset(train_df, train_transform)
val_ds = OcularDataset(val_df, eval_transform)
test_ds = OcularDataset(test_df, eval_transform)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)

# ------------------------------------------------------------
# 4. ResNet18 model
# ------------------------------------------------------------
weights = None
model = models.resnet18(weights=weights)
model.conv1 = nn.Conv2d(
    3, 64, kernel_size=7, stride=2, padding=3, bias=False
)
model.fc = nn.Linear(model.fc.in_features, len(classes))
model = model.to(DEVICE)

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss = criterion(logits, y)
            total_loss += loss.item() * y.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            total += y.size(0)

    return {
        "loss": total_loss / max(total, 1),
        "accuracy": correct / max(total, 1),
        "count": total
    }

# ------------------------------------------------------------
# 5. Train and save best checkpoint
# ------------------------------------------------------------
history = []
best_val_acc = -1.0
checkpoint_path = OUT / "NDCLD15_augmented_proxy_resnet18.pth"

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * y.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)

    train_loss = running_loss / max(total, 1)
    train_acc = correct / max(total, 1)
    val_metrics = evaluate(model, val_loader) if len(val_df) else {
        "loss": None, "accuracy": None, "count": 0
    }

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"]
    }
    history.append(row)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"train loss={train_loss:.4f}, train acc={train_acc:.4%} | "
        f"val loss={val_metrics['loss']}, "
        f"val acc={val_metrics['accuracy']}"
    )

    current_val = val_metrics["accuracy"]
    if current_val is None or current_val >= best_val_acc:
        best_val_acc = current_val if current_val is not None else best_val_acc
        state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
        torch.save({
            "model_state_dict": state,
            "num_classes": len(classes),
            "class_to_idx": class_to_idx,
            "proxy_label_warning": "Unverified filename-derived labels"
        }, checkpoint_path)

history_df = pd.DataFrame(history)
history_df.to_csv(OUT / "training_history.csv", index=False)

# ------------------------------------------------------------
# 6. Evaluate best checkpoint on held-out real images
# ------------------------------------------------------------
checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
base_model = model.module if isinstance(model, nn.DataParallel) else model
base_model.load_state_dict(checkpoint["model_state_dict"])
test_metrics = evaluate(model, test_loader)

print("\nHeld-out real-image proxy-label results:")
print(json.dumps(test_metrics, indent=2))

# ------------------------------------------------------------
# 7. Save plots and results
# ------------------------------------------------------------
plt.figure(figsize=(8, 4))
plt.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Train loss")
if history_df["val_loss"].notna().any():
    plt.plot(history_df["epoch"], history_df["val_loss"], marker="o", label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Notebook 10 — Training Curves")
plt.legend()
plt.tight_layout()
plt.savefig(OUT / "training_loss.png", dpi=150)
plt.close()

plt.figure(figsize=(8, 4))
plt.plot(history_df["epoch"], history_df["train_accuracy"], marker="o", label="Train accuracy")
if history_df["val_accuracy"].notna().any():
    plt.plot(history_df["epoch"], history_df["val_accuracy"], marker="o", label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Notebook 10 — Accuracy Curves (Proxy Labels)")
plt.legend()
plt.tight_layout()
plt.savefig(OUT / "training_accuracy.png", dpi=150)
plt.close()

# Save split manifest so the experiment can be reproduced.
train_df.assign(split="train").to_csv(OUT / "train_split.csv", index=False)
val_df.assign(split="validation").to_csv(OUT / "validation_split.csv", index=False)
test_df.assign(split="test").to_csv(OUT / "test_split.csv", index=False)

summary = {
    "notebook": 10,
    "dataset": "NDCLD15",
    "experiment": "exploratory filename-derived proxy-label recognition",
    "device": str(DEVICE),
    "gpu_count": torch.cuda.device_count(),
    "real_images_discovered": len(real_files),
    "traditional_images_available": len(trad_files),
    "gan_images_available_unlabelled": len(gan_files),
    "diffusion_images_available_unlabelled": len(diff_files),
    "proxy_label_classes": len(classes),
    "train_images": len(train_df),
    "validation_images": len(val_df),
    "test_images": len(test_df),
    "epochs": EPOCHS,
    "test_loss": test_metrics["loss"],
    "test_accuracy": test_metrics["accuracy"],
    "test_samples": test_metrics["count"],
    "verified_contact_lens_labels_available": False,
    "claim_limit": "Do not interpret proxy-label accuracy as valid contact-lens recognition accuracy."
}

with open(OUT / "Notebook10_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

conclusion = f"""# Notebook 10 — Recognition with Augmented Data: Conclusion

## Objective
Train an exploratory ResNet18 model using the real-image portion of the consolidated NDCLD15 dataset and evaluate it on held-out real images.

## Configuration
- Model: ResNet18
- Epochs: {EPOCHS}
- Input resolution: {IMAGE_SIZE} × {IMAGE_SIZE}
- Proxy classes: {len(classes)}
- Training images: {len(train_df)}
- Validation images: {len(val_df)}
- Test images: {len(test_df)}
- Device: {DEVICE}

## Results
- Test loss: {test_metrics['loss']:.4f}
- Test accuracy: {test_metrics['accuracy']:.4%}
- Test samples: {test_metrics['count']}

## Critical limitations
The labels used here are unverified filename-derived proxy labels. They are not verified contact-lens classes or verified biometric identities. Therefore, the reported test accuracy is only an exploratory proxy-label result and must not be presented as valid contact-lens recognition accuracy.

Although traditional augmentation files are available, this experiment uses only real images in its train/validation/test splits. It does not yet establish whether traditional, GAN, or diffusion augmentation improves performance. A valid comparison requires verified labels and identical, leakage-safe splits across experiments.

## Conclusion
The exploratory model training and held-out evaluation completed. Results are documented for reproducibility, but no claim about contact-lens classification performance or augmentation benefit can be made from this proxy-label experiment.
"""
(OUT / "Notebook10_Conclusion.md").write_text(conclusion, encoding="utf-8")

# ------------------------------------------------------------
# 8. ZIP results
# ------------------------------------------------------------
zip_path = PROJECT / "ND_Iris_Notebook10_Results.zip"
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in OUT.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=str(p.relative_to(OUT)))

with zipfile.ZipFile(zip_path, "r") as zf:
    bad_member = zf.testzip()
    archive_count = len(zf.namelist())

print("\n" + "=" * 65)
print("NOTEBOOK 10 — FINAL RESULTS")
print("=" * 65)
print(json.dumps(summary, indent=2))
print("Checkpoint:", checkpoint_path)
print("Conclusion:", OUT / "Notebook10_Conclusion.md")
print("Results ZIP:", zip_path)
print("Archived files:", archive_count)
print("ZIP integrity:", "PASSED" if bad_member is None else f"FAILED: {bad_member}")
print("Elapsed seconds:", round(time.time() - start_time, 2))


Device: cuda
GPU count: 2
Real: 6218
Traditional: 12467
GAN: 38
Diffusion: 17
Real images with proxy labels: 6216
Proxy classes: 322

Split sizes:
Train: 4213
Validation: 1000
Test: 1003
Epoch 1/3 | train loss=4.8451, train acc=18.9176% | val loss=4.350599484443665, val acc=0.226
Epoch 2/3 | train loss=3.6018, train acc=27.4626% | val loss=4.559429798126221, val acc=0.174
Epoch 3/3 | train loss=2.5844, train acc=39.6867% | val loss=2.818671081542969, val acc=0.355

Held-out real-image proxy-label results:
{
  "loss": 2.787205729860133,
  "accuracy": 0.36490528414755735,
  "count": 1003
}

NOTEBOOK 10 — FINAL RESULTS
{
  "notebook": 10,
  "dataset": "NDCLD15",
  "experiment": "exploratory filename-derived proxy-label recognition",
  "device": "cuda",
  "gpu_count": 2,
  "real_images_discovered": 6218,
  "traditional_images_available": 12467,
  "gan_images_available_unlabelled": 38,
  "diffusion_images_available_unlabelled": 17,
  "proxy_label_classes": 322,
  "train_images": 4213,
  "va

# Notebook 10 — Recognition with Augmented Data: Conclusion

## Objective
Train an exploratory ResNet18 recognition model using the consolidated NDCLD15 dataset and evaluate its performance on held-out real images.

## Experimental Configuration
- Model: ResNet18
- Hardware: 2 NVIDIA Tesla T4 GPUs
- Training epochs: 3
- Input resolution: 128 × 128 pixels
- Proxy classes: 322
- Training images: 4,213
- Validation images: 1,000
- Test images: 1,003

## Results
| Metric | Result |
|---|---:|
| Epoch 1 training accuracy | 18.92% |
| Epoch 2 training accuracy | 27.46% |
| Epoch 3 training accuracy | 39.69% |
| Final validation accuracy | 35.50% |
| Held-out test accuracy | 36.49% |
| Held-out test loss | 2.7872 |

## Interpretation
Training accuracy increased throughout the three epochs. The final held-out test accuracy was 36.49% on 1,003 real images.

The experiment used filename-derived proxy labels that were not verified contact-lens classes or verified biometric identities. Consequently, these metrics must be interpreted only as exploratory proxy-label results.

Although traditional, GAN, and diffusion images were available in the consolidated dataset, this experiment did not directly compare models trained with and without augmentation. It therefore does not establish an augmentation benefit.

## Limitations
- Verified contact-lens class labels were unavailable.
- Proxy labels may not correspond to meaningful recognition targets.
- Only three training epochs were completed.
- A controlled augmentation comparison requires verified labels and consistent, leakage-safe data splits.

## Conclusion
The exploratory ResNet18 training and held-out evaluation completed successfully. The checkpoint, training history, split manifests, plots, summary, and conclusion were archived in `ND_Iris_Notebook10_Results.zip`. These results document the experimental pipeline but do not demonstrate validated contact-lens recognition performance or improved accuracy from augmentation.